# AI.DETECT_ANOMALIES — BigQuery AI Functions

`AI.DETECT_ANOMALIES` is a table-valued function that detects anomalies in time series data using TimesFM. It forecasts expected values from historical data, then flags data points that deviate significantly.

**When to use it:**
- You want to detect unusual spikes or drops in time series data
- You need anomaly probability scores for each data point
- You want to detect anomalies across multiple time series simultaneously

**Alternatives:**
- [`AI.FORECAST`](../ai_forecast/) — Generate forecasts without anomaly detection
- [`AI.EVALUATE`](../ai_evaluate/) — Evaluate forecast accuracy instead of detecting anomalies

**Featured in:** [Time Series Intelligence](../../workflows/time_series_intelligence/)

**References:** [Full syntax reference](../../RESOURCES.md) | [Official documentation](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-detect-anomalies) | [Setup guide](../../setup/)

---
## Setup

Set your project and location, authenticate, and create a temporary dataset for this notebook.

> This function doesn't require a connection or model — it uses end-user credentials automatically. See the [Setup Reference](../../setup/) for details.

In [12]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [13]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm', 'bigframes')

In [14]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [15]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready
The bigquery_magics extension is already loaded. To reload it, use:
  %reload_ext bigquery_magics


---
## Examples — SQL

Progressive examples from simplest to most advanced. Each cell adds one new concept.

### Setup: Create historical and target data

Anomaly detection needs two inputs:
1. **Historical data** — used to build a forecast baseline
2. **Target data** — checked for anomalies against the baseline

We create data with a realistic pattern (upward trend + weekly seasonality), then inject two clear anomalies into the target period: a spike on Dec 15 and a drop on Dec 25.

In [16]:
# Create historical data (Jan-Nov 2024): upward trend + weekly seasonality + noise
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history` AS
WITH dates AS (
  SELECT date FROM UNNEST(GENERATE_DATE_ARRAY('2024-01-01', '2024-11-30')) AS date
)
SELECT date,
  GREATEST(0,
    1000
    + EXTRACT(DAYOFYEAR FROM date) * 2  -- upward trend (~2/day)
    + CASE EXTRACT(DAYOFWEEK FROM date)
        WHEN 1 THEN -200  -- Sunday dip
        WHEN 7 THEN 300   -- Saturday peak
        ELSE 0
      END
    + CAST(100 * (RAND() - 0.5) AS INT64)  -- noise
  ) AS daily_sales
FROM dates
'''
client.query(query).result()

# Create target data (Dec 2024): same pattern but with 2 injected anomalies
# IMPORTANT: target must follow the same pattern as history, otherwise
# the model flags every deviation from the learned seasonality.
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target` AS
WITH dates AS (
  SELECT date FROM UNNEST(GENERATE_DATE_ARRAY('2024-12-01', '2024-12-31')) AS date
)
SELECT date,
  CASE
    WHEN date = '2024-12-15' THEN 5000  -- injected spike anomaly (Sunday)
    WHEN date = '2024-12-25' THEN 50    -- injected drop anomaly (Wednesday)
    ELSE GREATEST(0,
      1000
      + EXTRACT(DAYOFYEAR FROM date) * 2  -- same trend
      + CASE EXTRACT(DAYOFWEEK FROM date)
          WHEN 1 THEN -200  -- same weekly seasonality
          WHEN 7 THEN 300
          ELSE 0
        END
      + CAST(100 * (RAND() - 0.5) AS INT64)  -- same noise
    )
  END AS daily_sales
FROM dates
'''
client.query(query).result()
print('Historical and target data created')

Historical and target data created


### 1. Basic anomaly detection

Pass historical and target tables. Returns each target point with `is_anomaly`, bounds, and probability.

In [17]:
query = f'''
SELECT
  time_series_timestamp,
  time_series_data,
  is_anomaly,
  ROUND(anomaly_probability, 4) AS anomaly_prob,
  ROUND(lower_bound, 0) AS lower_bound,
  ROUND(upper_bound, 0) AS upper_bound
FROM AI.DETECT_ANOMALIES(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history`,
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target`,
  data_col => 'daily_sales',
  timestamp_col => 'date'
)
ORDER BY time_series_timestamp
'''
df = client.query(query).to_dataframe()

anomalies = df[df['is_anomaly'] == True]
print(f'Total points: {len(df)}, Anomalies detected: {len(anomalies)}')
print(f'\nAnomalous points:')
print(anomalies[['time_series_timestamp', 'time_series_data', 'anomaly_prob', 'lower_bound', 'upper_bound']].to_string(index=False))

Total points: 31, Anomalies detected: 2

Anomalous points:
    time_series_timestamp  time_series_data  anomaly_prob  lower_bound  upper_bound
2024-12-15 00:00:00+00:00            5000.0           1.0       1424.0       1564.0
2024-12-25 00:00:00+00:00              50.0           1.0       1613.0       1770.0


### 2. Adjusting the anomaly threshold

The `anomaly_prob_threshold` controls sensitivity (default: 0.95). A data point is flagged as anomalous if its anomaly probability exceeds this threshold.

- **Higher threshold** (e.g., 0.99) → fewer, more extreme anomalies only
- **Lower threshold** (e.g., 0.8) → more anomalies, catches subtler deviations

Here we show all points sorted by anomaly probability to see the full distribution.

In [18]:
query = f'''
SELECT
  time_series_timestamp,
  time_series_data,
  is_anomaly,
  ROUND(anomaly_probability, 4) AS anomaly_prob,
  ROUND(lower_bound, 0) AS lower,
  ROUND(upper_bound, 0) AS upper
FROM AI.DETECT_ANOMALIES(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history`,
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target`,
  data_col => 'daily_sales',
  timestamp_col => 'date',
  anomaly_prob_threshold => 0.99  -- stricter: only the most extreme anomalies
)
ORDER BY anomaly_prob DESC
'''
df = client.query(query).to_dataframe()

anomalies = df[df['is_anomaly'] == True]
print(f'With threshold=0.99: {len(anomalies)} anomalies (vs default 0.95)')
df.head(10)

With threshold=0.99: 2 anomalies (vs default 0.95)


,time_series_timestamp,time_series_data,is_anomaly,anomaly_prob,lower,upper
0,2024-12-15 00:00:00+00:00,5000.0,True,1.0000,1405.0,1590.0
1,2024-12-25 00:00:00+00:00,50.0,True,1.0000,1589.0,1796.0
2,2024-12-18 00:00:00+00:00,1745.0,False,0.9425,1590.0,1770.0
3,2024-12-20 00:00:00+00:00,1754.0,False,0.9310,1608.0,1779.0
4,2024-12-27 00:00:00+00:00,1759.0,False,0.9099,1612.0,1793.0
5,2024-12-09 00:00:00+00:00,1732.0,False,0.9092,1588.0,1766.0
6,2024-12-08 00:00:00+00:00,1534.0,False,0.8838,1383.0,1574.0
7,2024-12-26 00:00:00+00:00,1755.0,False,0.8688,1605.0,1799.0
8,2024-12-04 00:00:00+00:00,1711.0,False,0.8675,1584.0,1744.0
9,2024-12-23 00:00:00+00:00,1743.0,False,0.8645,1587.0,1784.0


### 3. Using a query for history (subset)

Pass a query instead of a table reference to filter historical data.

In [19]:
query = f'''
SELECT
  time_series_timestamp,
  time_series_data,
  is_anomaly,
  ROUND(anomaly_probability, 4) AS anomaly_prob
FROM AI.DETECT_ANOMALIES(
  (SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history`
   WHERE date >= '2024-09-01'),  -- use only recent history
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target`,
  data_col => 'daily_sales',
  timestamp_col => 'date'
)
WHERE is_anomaly = TRUE
ORDER BY anomaly_prob DESC
'''
client.query(query).to_dataframe()

,time_series_timestamp,time_series_data,is_anomaly,anomaly_prob
0,2024-12-15 00:00:00+00:00,5000.0,True,1.0
1,2024-12-25 00:00:00+00:00,50.0,True,1.0


---
## Examples — `%%bigquery` Magics

The same examples using IPython magic commands. Magics let you write SQL directly in notebook cells without Python string wrapping.

Key patterns:
- `%%bigquery` — run SQL, display results inline
- `%%bigquery df` — run SQL, capture results into a pandas DataFrame

### Anomaly detection with `%%bigquery`

In [20]:
%%bigquery --project {PROJECT_ID}

SELECT time_series_timestamp, time_series_data, is_anomaly,
  ROUND(anomaly_probability, 4) AS anomaly_prob
FROM AI.DETECT_ANOMALIES(
  TABLE `statmike-mlops-349915.bq_ai_functions.ai_detect_anomalies_history`,
  TABLE `statmike-mlops-349915.bq_ai_functions.ai_detect_anomalies_target`,
  data_col => 'daily_sales',
  timestamp_col => 'date'
)
WHERE is_anomaly = TRUE
ORDER BY anomaly_probability DESC

Query is running:   0%|          |

Downloading:   0%|          |

,time_series_timestamp,time_series_data,is_anomaly,anomaly_prob
0,2024-12-15 00:00:00+00:00,5000.0,True,1.0
1,2024-12-25 00:00:00+00:00,50.0,True,1.0


---
## Examples — BigFrames

`AI.DETECT_ANOMALIES` has no direct BigFrames equivalent for TimesFM. Use `session.read_gbq_query()` to execute the SQL from BigFrames.

**Note:** `bigframes.ml.forecasting.ARIMAPlus.detect_anomalies()` exists but uses ARIMA_PLUS, not TimesFM.

In [21]:
import bigframes.pandas as bpd

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION

### Running AI.DETECT_ANOMALIES via read_gbq_query

In [22]:
query = f"""
SELECT time_series_timestamp, time_series_data, is_anomaly, anomaly_probability
FROM AI.DETECT_ANOMALIES(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history`,
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target`,
  data_col => 'daily_sales',
  timestamp_col => 'date'
)
ORDER BY time_series_timestamp
"""
df = bpd.read_gbq_query(query)
df.to_pandas()

,time_series_timestamp,time_series_data,is_anomaly,anomaly_probability
0,2024-12-01 00:00:00+00:00,1432.0,False,0.420774
1,2024-12-02 00:00:00+00:00,1693.0,False,0.791753
2,2024-12-03 00:00:00+00:00,1688.0,False,0.625538
3,2024-12-04 00:00:00+00:00,1711.0,False,0.867538
4,2024-12-05 00:00:00+00:00,1657.0,False,0.002025
5,2024-12-06 00:00:00+00:00,1654.0,False,0.202802
6,2024-12-07 00:00:00+00:00,1961.0,False,0.080202
7,2024-12-08 00:00:00+00:00,1534.0,False,0.883816
8,2024-12-09 00:00:00+00:00,1732.0,False,0.909245
9,2024-12-10 00:00:00+00:00,1718.0,False,0.847772


---
## Cleanup

Drop resources created by this notebook. Shared resources (dataset, models, connection) are left for other notebooks.

In [ ]:
client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_history', not_found_ok=True)
print('Table ai_detect_anomalies_history deleted')
client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.ai_detect_anomalies_target', not_found_ok=True)
print('Table ai_detect_anomalies_target deleted')


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')